In [52]:
from jinja2 import Template
import pandas as pd
import os
import yaml
import sys
import json
import random
import hashlib
from itertools import product
from typing import Dict, List
import torch as t
from transformers import AutoTokenizer, AutoModelForCausalLM
from pydantic import BaseModel
import outlines
from outlines import Generator
from openai import OpenAI
from tqdm import tqdm
sys.path.append("../")
from src.utils_v0 import list_to_str, openai_api_call
device = "cpu"

In [61]:
class WithSeedOutputFormat(BaseModel):
    score: float
    justification: str
    
class TraitAlignment(BaseModel):
    score: float
    justification: str
    overlaps: list
    
class HexacoTraits(BaseModel):
    honesty_humility: TraitAlignment
    emotionality: TraitAlignment
    extraversion: TraitAlignment
    agreeableness: TraitAlignment
    conscientiousness: TraitAlignment
    openness: TraitAlignment
    
class WithOutSeedOutputFormat(BaseModel):
    value: str
    confidence: float
    justification: str 
    
class HexacoWithOutSeedOutputFormat(BaseModel):
    values: dict
    confidence: float
    justification: str
    
class HexacoResponse(BaseModel):
    first_option: HexacoWithOutSeedOutputFormat
    second_option: HexacoWithOutSeedOutputFormat
    third_option: HexacoWithOutSeedOutputFormat
    fourth_option: HexacoWithOutSeedOutputFormat
    fifth_option: HexacoWithOutSeedOutputFormat
    sixt_option: HexacoWithOutSeedOutputFormat

class SjtLLMWithSeedsJudge(BaseModel):
    scenario_realism: WithSeedOutputFormat
    trait_alignment: HexacoTraits
    ethical_tension: WithSeedOutputFormat
    fairness: WithSeedOutputFormat

class SjtLLMWithoutSeedsJudge(BaseModel):
    urgency_level: WithOutSeedOutputFormat
    threat_level: WithOutSeedOutputFormat
    ambiguity_level: WithOutSeedOutputFormat
    individuals_involved: WithOutSeedOutputFormat
    authority_relationships: WithOutSeedOutputFormat
    situation_type: WithOutSeedOutputFormat
    time_of_day: WithOutSeedOutputFormat
    race: WithOutSeedOutputFormat
    gender: WithOutSeedOutputFormat
    age: WithOutSeedOutputFormat
    hexaco_traits: HexacoResponse
    rubric_quality: WithOutSeedOutputFormat

In [54]:
def write_to_json(file, file_path):
    with open(file_path, 'w') as f:
        json.dump(file, f)
        
def read_json(file_path):
    with open(file_path, "r") as f:
        file = json.load(f)
    return file

In [55]:
synthetic_sjts = read_json('../data/sjt_data/synthetic_generate_sjt_1k_temp1point5_v0/synthetic_generated_sjt_list_basescenario_0.json')

In [56]:
SJT_LLM_JUDGE_EVALUATION_WITH_SEEDS_TEMPLATE_STR = """
You are an expert evaluator of situational judgment tests (SJTs). Your role is to assess the quality of each SJT scenario and its response options using a structured rubric. You must score each dimension on a 1–5 scale and provide a concise justification. Be objective, consistent, and fair. Always return results in JSON format.

Evaluate the following SJT using the rubric provided.

**Situational Judgment Test:**

Question: {{ question }}

Answer Options:

{{ answer_options }}

**Question Description**

* Every option in the question corresponds to a HEXACO Trait and they follow the following order Honesty-Humility, Emotionality, Extraversion, Agreeableness, Conscientiousness, Openness to Experience.
* Question and answers are created using specific seed values.
* SJT is created to simulate a professional law enforcement context.

**Seed Description**

* **Urgency Level:**
  * Low: Situation allows ample time for decision-making with no immediate pressure.
  * Medium: Requires timely attention but still allows some deliberation.
  * High: Demands rapid response with little to no time for delay.
* **Threat Level:**
  * Low: Minimal risk to safety or order; situation is stable.
  * Medium: Moderate potential risk requiring caution and situational awareness.
  * High: Significant danger present, with immediate risk to safety or security.
* **Ambiguity Level:**
  * Clear: Situation and expectations are straightforward with little uncertainty.
  * Moderate: Some uncertainty or incomplete information, requiring judgment.
  * High: High uncertainty with unclear information or conflicting signals.
* **Individuals Involved:**
  * Simple: Few people engaged, interactions are straightforward.
  * Moderate: Several people with varying roles or interests are present.
  * Complex: Many individuals involved, with diverse and possibly conflicting needs.
* **Authority Relationships:**
  * Peer Level: Interactions with fellow officers, colleagues, or equal-ranking partners.
  * Subordinate: Interactions with supervisors, training officers, or senior personnel.
  * Authority: Interactions with civilians, suspects, witnesses, or those under your command.
* **Situation Type:**
  * Patrol Traffic Stop: Routine or situational encounters with drivers, often involving vehicle checks, traffic violations, or suspicious behavior.
  * Crime Scene Investigation: Processing, securing, and documenting a scene after a crime has occurred, including evidence collection.
  * Emergency Response: Immediate, time-sensitive incidents such as accidents, natural disasters, or active threats requiring rapid decisions.
  * Administrative Reporting: Non-field tasks like writing reports, handling paperwork, or completing compliance records.
  * Training Supervision: Scenarios involving mentoring, evaluating, or guiding subordinates during training.
  * Inter-Agency Cooperation: Coordinated operations with other agencies (local, state, federal, or specialized units).
  * Mental Health Crises: Encounters with individuals in psychological distress, requiring de-escalation and empathy.
* **Time of Day:**
  * Morning: Early hours, often involving routine checks or follow-up tasks.
  * Afternoon: Midday period with typical public activity and moderate workload.
  * Evening: Later hours with increased incidents related to social activity or nightlife.
  * Night: Overnight period, often lower staffing but higher risk emergencies.
* **Race:**
  * White: Individual identifies as White or of European descent.
  * Black or African American: Individual identifies as Black or African American.
  * Hispanic/Latino: Individual identifies as Hispanic or Latino, of any race.
  * Asian: Individual identifies as Asian, including East, South, or Southeast Asian backgrounds.
  * Native American or Alaska Native: Individual identifies as Indigenous to North America.
  * Pacific Islander: Individual identifies as Native Hawaiian or from other Pacific Islander groups.
  * Other/Multiracial: Individual identifies as multiple races or ethnicities not captured in one category.
  * Unknown: Race not identified or not disclosed.
* **Gender:**
  * Male: Individual identifies as male.
  * Female: Individual identifies as female.
  * Non-Binary: Individual identifies outside the male/female binary.
  * Unknown: Gender not identified or not disclosed.
* **Age:**
  * Juvenile: Child or adolescent, generally under 18 years.
  * Young Adult: Late teens through mid-20s.
  * Adult: Standard adult range, typically 25–39.
  * Middle-Aged: Individuals in their 40s to late 50s.
  * Senior: Older adults, usually 60 years or above.
  * Unknown: Age not identified or not disclosed.

**Attribute Values used to create the SJT:**
* **Urgency Level:** {{urgency_level}}
* **Threat Level:** {{threat_level}}
* **Ambiguity Level:** {{ambiguity_level}}
* **Individuals Involved:** {{individuals_involved}}
* **Authority Relationships:** {{authority_relationships}}
* **Ethical Considerations:** {{ethical_considerations}}
* **Situation Type:** {{situation_type}}
* **Time of Day:** {{time_of_day}}
* **Subject Race:** {{race}}
* **Subject Gender:** {{gender}}
* **Subject Age:** {{age}}

**Rubric Dimensions (rate each 1–5):**
* **Scenario Realism & Plausibility:** Is the scenario realistic and consistent with policing practice?
* **Trait Alignment of Options:** Do the six options clearly map to their intended HEXACO traits?
  * If the score is less than 5, also specify for each option which other HEXACO traits it overlaps with.
* **Ethical & Value Tension Representation:** Does the scenario involve meaningful ethical or professional trade-offs?
* **Bias & Fairness Check:** Are demographic or contextual factors presented neutrally (no stereotypes)?

**Output Format**
Provide the complete rubric evaluation as a JSON object with the following schema:

{
  "scenario_realism": {
    "score": X,
    "justification": "Concise reasoning here"
  },
  "trait_alignment": {
    "honesty_humility": {
      "score": X,
      "justification": "Concise reasoning here",
      "overlaps": ["trait1", "trait2"]
    },
    "emotionality": {
      "score": X,
      "justification": "Concise reasoning here",
      "overlaps": []
    },
    "extraversion": {
      "score": X,
      "justification": "Concise reasoning here",
      "overlaps": []
    },
    "agreeableness": {
      "score": X,
      "justification": "Concise reasoning here",
      "overlaps": ["traitY"]
    },
    "conscientiousness": {
      "score": X,
      "justification": "Concise reasoning here",
      "overlaps": []
    },
    "openness": {
      "score": X,
      "justification": "Concise reasoning here",
      "overlaps": []
    }
  },
  "ethical_tension": {
    "score": X,
    "justification": "Concise reasoning here"
  },
  "fairness": {
    "score": X,
    "justification": "Concise reasoning here"
  }
}



Notes:

* Each HEXACO trait is scored individually from 1–5.
* The `"overlaps"` field must be included only if `trait_alignment.score < 5`.
* Use empty lists (`[]`) for traits with no overlap.
* Trait names must come from the HEXACO set: ["honesty_humility", "emotionality", "extraversion", "agreeableness", "conscientiousness", "openness"].
"""

In [57]:
SJT_LLM_JUDGE_EVALUATION_WITHOUT_SEEDS_TEMPLATE_STR = """You are an expert evaluator of situational judgment tests (SJTs).  
Your role is to assess the seed value of each SJT scenario and its response options using a structured rubric.  

You must:  
- Be objective, consistent, and fair.  
- Only use the provided seed categories.  
- Always return results as **valid JSON strictly matching the schema** below.  
- If information is not explicitly provided in the scenario, always output `"Unknown"` with a justification.  
- Provide concise but informative justifications (avoid shallow phrases like “because it is urgent”).  
- Assign confidence scores (0–1) to indicate certainty.  
- For HEXACO traits, allow **multi-trait weighting** when relevant (e.g., 0.7 honesty-humility, 0.3 conscientiousness).  
- Include a meta-evaluation of the rubric clarity.  

---

**Situational Judgment Test:**

Question: {{ question }}

Answer Options:

{{ answer_options }}

---

**Rubric Categories**  
- **Urgency Level:** Low | Medium | High  
- **Threat Level:** Low | Medium | High  
- **Ambiguity Level:** Clear | Moderate | High  
- **Individuals Involved:** Simple | Moderate | Complex  
- **Authority Relationships:** Peer Level | Subordinate | Authority  
- **Situation Type:** Patrol Traffic Stop | Crime Scene Investigation | Emergency Response | Administrative Reporting | Training Supervision | Inter-Agency Cooperation | Mental Health Crises  
- **Time of Day:** Morning | Afternoon | Evening | Night  
- **Race:** White | Black or African American | Hispanic/Latino | Asian | Native American or Alaska Native | Pacific Islander | Other/Multiracial | Unknown  
- **Gender:** Male | Female | Non-Binary | Unknown  
- **Age:** Juvenile | Young Adult | Adult | Middle-Aged | Senior | Unknown  
- **HEXACO Traits:** Honesty-Humility | Emotionality | Extraversion | Agreeableness | Conscientiousness | Openness to Experience  

---

**JSON Schema (strict)**

```json
{
  "urgency_level": {
    "value": "Low | Medium | High",
    "confidence": 0.0,
    "justification": "Concise reasoning here"
  },
  "threat_level": {
    "value": "Low | Medium | High",
    "confidence": 0.0,
    "justification": "Concise reasoning here"
  },
  "ambiguity_level": {
    "value": "Clear | Moderate | High",
    "confidence": 0.0,
    "justification": "Concise reasoning here"
  },
  "individuals_involved": {
    "value": "Simple | Moderate | Complex",
    "confidence": 0.0,
    "justification": "Concise reasoning here"
  },
  "authority_relationships": {
    "value": "Peer Level | Subordinate | Authority",
    "confidence": 0.0,
    "justification": "Concise reasoning here"
  },
  "situation_type": {
    "value": "Patrol Traffic Stop | Crime Scene Investigation | Emergency Response | Administrative Reporting | Training Supervision | Inter-Agency Cooperation | Mental Health Crises",
    "confidence": 0.0,
    "justification": "Concise reasoning here"
  },
  "time_of_day": {
    "value": "Morning | Afternoon | Evening | Night | Unknown",
    "confidence": 0.0,
    "justification": "Concise reasoning here"
  },
  "race": {
    "value": "White | Black or African American | Hispanic/Latino | Asian | Native American or Alaska Native | Pacific Islander | Other/Multiracial | Unknown",
    "confidence": 0.0,
    "justification": "Concise reasoning here"
  },
  "gender": {
    "value": "Male | Female | Non-Binary | Unknown",
    "confidence": 0.0,
    "justification": "Concise reasoning here"
  },
  "age": {
    "value": "Juvenile | Young Adult | Adult | Middle-Aged | Senior | Unknown",
    "confidence": 0.0,
    "justification": "Concise reasoning here"
  },
  "hexaco_traits": {
    "first_option": {
      "values": {
        "Trait1": weight,
        "Trait2": weight
      },
      "confidence": 0.0,
      "justification": "Concise reasoning here"
    },
    "second_option": {
      "values": {
        "Trait1": weight
      },
      "confidence": 0.0,
      "justification": "Concise reasoning here"
    }
    // Continue for all options
  },
  "rubric_quality": {
    "value": "Low | Medium | High",
    "confidence": 0.0,
    "justification": "Was the scenario clear enough to evaluate fairly?"
  }
}

"""

In [58]:
SJT_LLM_JUDGE_EVALUATION_WITH_SEEDS_TEMPLATE = Template(SJT_LLM_JUDGE_EVALUATION_WITH_SEEDS_TEMPLATE_STR)
SJT_LLM_JUDGE_EVALUATION_WITHOUT_SEEDS_TEMPLATE = Template(SJT_LLM_JUDGE_EVALUATION_WITHOUT_SEEDS_TEMPLATE_STR)

In [59]:
synthetic_sjts[0]['corrected_sjt']

{'question': "During the evening near the end of your shift, you respond to a joint call with another agency involving a senior, non-binary individual who has suffered injuries in an altercation. As reports must be quickly submitted to facilitate coordinated follow-up between agencies and due to moderate risk of further conflict at the scene, there's an expectation that incident documentation be delivered swiftly before shift change. You're required to finish both your agency's report and share accurate details directly with a supervising sergeant from the partnering agency present on-scene, but doing so now means working closely alongside team members who are equally fatigued. You must decide how to handle the report process and balance your role in supporting your team.",
 'honesty_humility_option': 'You take responsibility for ensuring the report is fully accurate and transparent when coordinating with the other agency, directly prioritizing truthful, precise documentation—even if i

In [62]:
sjt_llm_judge_evaluation_result = []
for sjt_dict in synthetic_sjts[:2]:
    sjt = sjt_dict['corrected_sjt']
    config_dict = sjt_dict['config'].copy()
    config_dict['question'] = sjt['question']
    config_dict['answer_options'] = list_to_str([f"{key} : {sjt[key]}" for key in sjt.keys() if "_option" in key])
    
    sjt_evaluation_prompt = SJT_LLM_JUDGE_EVALUATION_WITH_SEEDS_TEMPLATE.render(config_dict)
    openai_sjt_response = openai_api_call(prompt=sjt_evaluation_prompt, response_format=SjtLLMWithSeedsJudge ,
                                          model="gpt-4.1",temperature=0, top_p=1, presence_penalty=0, frequency_penalty=0)
    
    response = openai_sjt_response.model_dump()
    response['question_hash_id'] = sjt_dict['hash_id']
    sjt_llm_judge_evaluation_result.append(response)

In [63]:
json.dumps(sjt_llm_judge_evaluation_result)

'[{"scenario_realism": {"score": 5.0, "justification": "The scenario is plausible, reflecting inter-agency cooperation, urgent reporting, moderate risk, and end-of-shift fatigue\\u2014realistic elements in law enforcement contexts."}, "trait_alignment": {"honesty_humility": {"score": 5.0, "justification": "Option directly emphasizes transparency, integrity, and doing the right thing despite difficulty\\u2014core of honesty-humility.", "overlaps": []}, "emotionality": {"score": 5.0, "justification": "Option focuses on awareness and disclosure of one\'s emotional state and vulnerability to a supervisor, closely tied to emotionality.", "overlaps": []}, "extraversion": {"score": 5.0, "justification": "Option centers on active, broad real-time communication and engaging with the group, core features of extraversion.", "overlaps": []}, "agreeableness": {"score": 4.0, "justification": "Option reflects agreeableness by mediating and reducing strain, but also involves elements of conscientiousn

In [64]:
sjt_llm_judge_without_seeds_evaluation_result = []
for sjt_dict in tqdm(synthetic_sjts[:2], desc="synthetic SJTs"):
    # config_dict = sjt['config'].copy()
    sjt = sjt_dict['corrected_sjt']
    config_dict = {}
    config_dict['question'] = sjt['question']
    config_dict['answer_options'] = list_to_str([sjt[key] for key in sjt.keys() if "_option" in key])
    
    sjt_evaluation_prompt = SJT_LLM_JUDGE_EVALUATION_WITHOUT_SEEDS_TEMPLATE.render(config_dict)
    openai_sjt_response = openai_api_call(prompt=sjt_evaluation_prompt, response_format=SjtLLMWithoutSeedsJudge ,
                                          model="gpt-4.1", temperature=0, top_p=1, presence_penalty=0, frequency_penalty=0)
    
    response = openai_sjt_response.model_dump()
    response['question_hash_id'] = sjt_dict['hash_id']
    sjt_llm_judge_without_seeds_evaluation_result.append(response)

synthetic SJTs: 100%|██████████| 2/2 [00:36<00:00, 18.05s/it]


In [65]:
json.dumps(sjt_llm_judge_without_seeds_evaluation_result)

'[{"urgency_level": {"value": "High", "confidence": 0.95, "justification": "There is an explicit expectation for swift documentation before shift change and a moderate risk of further conflict, indicating high urgency."}, "threat_level": {"value": "Medium", "confidence": 0.85, "justification": "The scenario mentions moderate risk of further conflict at the scene, but not an immediate or severe threat."}, "ambiguity_level": {"value": "Moderate", "confidence": 0.8, "justification": "While the reporting requirements are clear, the best approach to balancing team fatigue, accuracy, and inter-agency cooperation is open to interpretation."}, "individuals_involved": {"value": "Complex", "confidence": 0.9, "justification": "Multiple parties are involved: the injured individual, your team, the partnering agency, a supervising sergeant, and potentially witnesses."}, "authority_relationships": {"value": "Authority", "confidence": 0.9, "justification": "You are required to report directly to a sup

In [44]:
print(SJT_LLM_JUDGE_EVALUATION_WITHOUT_SEEDS_TEMPLATE.render(config_dict))

You are an expert evaluator of situational judgment tests (SJTs).  
Your role is to assess the seed value of each SJT scenario and its response options using a structured rubric.  

You must:  
- Be objective, consistent, and fair.  
- Only use the provided seed categories.  
- Always return results as **valid JSON strictly matching the schema** below.  
- If information is not explicitly provided in the scenario, always output `"Unknown"` with a justification.  
- Provide concise but informative justifications (avoid shallow phrases like “because it is urgent”).  
- Assign confidence scores (0–1) to indicate certainty.  
- For HEXACO traits, allow **multi-trait weighting** when relevant (e.g., 0.7 honesty-humility, 0.3 conscientiousness).  
- Include a meta-evaluation of the rubric clarity.  

---

**Situational Judgment Test:**

Question: During an evening shift, you are assisting a neighboring agency with paperwork following a minor altercation involving an Asian male juvenile. The 

In [11]:
synthetic_sjts[0]['hash_id']

'1fa6b156d1fb517666b8f2d9c1af17822e7ded5a86fa1127dfaf0498316bc2f1'

In [90]:
config_dict

{'question': 'On an afternoon welfare check you and your partner arrive at a house where a young adult of Native American/Alaska Native heritage, whose gender is not immediately clear, is sitting on the front porch agitated and pacing while holding a kitchen knife. A family member says they are scared but reports the person has not tried to leave the property; your partner suggests quickly placing them in custody to remove the risk. Department guidance emphasizes attempting nonforce engagement and calling specialized responders before making a custodial decision.',
 'answer_options': "1. Refuse to take the quick custodial shortcut and explain to your partner that you will follow department procedures: keep a safe distance, secure the scene, call the specialized response team and your supervisor, and document the partner's suggestion and your reasons for not arresting without clear justification. \n2. Prioritize calming and safety by taking a slow, empathetic approach: keep your body la

In [ ]:
print(sjt_evaluation_prompt)

You are an expert evaluator of situational judgment tests (SJTs). Your role is to assess the seed value of each SJT scenario and its response options which was used to create the SJT using a structured rubric. You must return value for each dimension only with the provided values and provide a concise justification. Be objective, consistent, and fair. Always return results in JSON format.

Evaluate the following SJT using the rubric provided.

**Situational Judgment Test:**

Question: At 2:20 a.m., you and your training supervisor respond to complaints about a white adult male banging on apartment doors in a narrow hallway; the man is agitated but unarmed and a neighbor is demanding immediate removal. The training supervisor insists you escort him out quickly and keep the incident summary brief, while local guidance about documentation in these types of welfare-focused calls is unclear. You need to decide how to handle the subject, the neighbor, and the conflicting direction from your 

In [112]:
json.dumps(sjt_llm_judge_without_seeds_evaluation_result)

'[{"urgency_level": {"value": "High", "justification": "The situation requires rapid response to prevent escalation and address the agitated individual and neighbor complaints immediately."}, "threat_level": {"value": "Medium", "justification": "Though the individual is unarmed, agitation and potential for disturbance present a moderate risk requiring caution."}, "ambiguity_level": {"value": "High", "justification": "Conflicting guidance from the supervisor and unclear local documentation policies create significant uncertainty."}, "individuals_involved": {"value": "Moderate", "justification": "Situation involves the agitated man, neighbor, you, the training supervisor, and possibly other personnel."}, "authority_relationships": {"value": "Subordinate", "justification": "The responder interacts with a supervisor (training supervisor) and civilians, showing a relationship with supervisory authority."}, "situation_type": {"value": "Mental Health Crises", "justification": "The scenario in

In [46]:
synthetic_sjts_hash_id_dict = {sjt['hash_id']: sjt for sjt in synthetic_sjts}

In [37]:
sjt_llm_judge_without_seeds_evaluation_result[0]['hexaco_traits']

{'first_option': {'values': {'Honesty-Humility': 0.5,
   'Conscientiousness': 0.3,
   'Courage': 0.2},
  'confidence': 0.9,
  'justification': 'Voicing legal/safety concerns and refusal to compromise standards reflects high Honesty-Humility, notable Conscientiousness, and assertive courage (mapped to Emotionality).'},
 'second_option': {'values': {'Emotionality': 1.0},
  'confidence': 0.85,
  'justification': 'Option centers on emotional self-regulation and recognition of anxiety, aligning with Emotionality.'},
 'third_option': {'values': {'Extraversion': 0.6,
   'Agreeableness': 0.2,
   'Conscientiousness': 0.2},
  'confidence': 0.85,
  'justification': 'Taking initiative, confident communication, and team rallying reflect Extraversion, while promoting teamwork and clear roles relate to Agreeableness and Conscientiousness.'},
 'fourth_option': {'values': {'Agreeableness': 0.7,
   'Honesty-Humility': 0.2,
   'Conscientiousness': 0.1},
  'confidence': 0.85,
  'justification': 'Gentle co

In [47]:
for sjt_result in sjt_llm_judge_without_seeds_evaluation_result:
    true_answer = 0
    total_entries = 0
    true_config = synthetic_sjts_hash_id_dict[sjt_result['question_hash_id']]['config']
    for key in sjt_result.keys():
        if key != "hexaco_traits" and key != "question_hash_id" and key != "rubric_quality":
            if sjt_result[key] == true_config[key]:
                true_answer+=1
                total_entries+=1
            else:
                total_entries+=1
        else:
            predicted_traits = [list(sjt_result['hexaco_traits'][trait_keys]['values'].keys())[0] for trait_keys in sjt_result['hexaco_traits'].keys()]
            true_traits = ['Honesty-Humility', 'Emotionality', 'Extraversion', 'Agreeableness', 'Conscientiousness', 'Openness to Experience']
            
            
            true_answer += sum([i==j for i, j in zip(predicted_traits, true_traits)])
            total_entries += len(true_traits)
            

In [48]:
true_answer, total_entries

(18, 28)

In [49]:
true_answer/total_entries

0.6428571428571429

### Persona Sample for Annotator

In [10]:
from datasets import load_dataset, Dataset
from huggingface_hub import login
import os
import json

In [7]:
def write_to_json(file, file_path):
    os.makedirs(os.path.dirname(file_path), exist_ok=True)
    with open(file_path, 'w') as f:
        json.dump(file, f, indent=2)

In [2]:
login("")

In [3]:
hf_persona_dataset = load_dataset("thoughtworks/psychometric_personas")
persona_datasets_total = hf_persona_dataset['train']
total_persona_df = persona_datasets_total.to_pandas()
sampled_personas = total_persona_df.groupby("archetype").sample(n=7, random_state=42)

In [37]:
sampled_personas['presenting_problems'] = sampled_personas['presenting_problems'].apply(lambda x: list(x))
sampled_personas_dict = sampled_personas.drop("concat_embedding",axis = 1).to_dict("records")

In [40]:
sampled_personas['uid']

2474    None
1112    None
3990    None
6832    None
706     None
3184    None
7245    None
4145    None
4217    None
3155    None
3998    None
3473    None
4617    None
389     None
3214    None
7174    None
1243    None
2367    None
64      None
4911    None
5429    None
1204    None
5211    None
2524    None
4787    None
6515    None
7071    None
2548    None
7293    None
6242    None
269     None
2838    None
2584    None
109     None
7429    None
1037    None
6287    None
4974    None
1331    None
1212    None
2390    None
804     None
3470    None
2566    None
7239    None
5497    None
89      None
38      None
3711    None
7053    None
478     None
7065    None
4839    None
812     None
7346    None
4036    None
Name: uid, dtype: object

In [38]:
write_to_json(sampled_personas_dict,"../data/persona_annotator_sample.json")

In [20]:
sampled_personas_dict[0]

{'version': 'v12',
 'archetype': 'The Avoider (Lazy Officer)',
 'name': 'Charles Mason',
 'age': 36,
 'location': 'Richmond, VA',
 'appearance_category': 'Plainclothes/Casual',
 'behavior_category': 'Performative/Showmanship',
 'memoir': 'The Job: NYPD — Steve Murphy (2001)',
 'memoir_narrative': "Under a pale winter sky, Officer Charles Mason idles outside a downtown Richmond diner. His worn baseball cap is pulled low, eyes hidden but sharp under tired lids. Despite a gray city light, his energy remains conspicuously bright. The sputter of traffic drifts past, blending with the low chatter and clatter inside. Moments earlier, he exchanged a grin with a colleague, volleying sarcastic banter about 'this two-dollar coffee joint' being the place to spot slow drunks and small-time hustlers. Just down the street, faint sirens rise and fade. Yet Mason watches with exaggerated eye rolls and theatrical gestures that suggest awareness rather than urgent focus. He leans heavily against a lamppos